# Tradução individual para modelos Transformer v80+

Notebook para testar automaticamente todos os checkpoints Transformer a partir de `v80` usando a arquitetura atual do projeto.

- Carrega o tokenizer mais recente disponível
- Lê o config do modelo correspondente
- Instancia `Transformer` com `kv_cache=True`
- Testa várias frases em cada checkpoint
- Exibe a tradução de cada modelo

In [ ]:
import json
import os

import torch
import tokenizers

from models.transformer import Transformer

print("CUDA disponível:", torch.cuda.is_available())

In [ ]:
PATH_MODELS = "artifacts"
PATH_CONFIGS = "configs"


def resolve_tokenizer_path():
    candidates = [
        "artifacts/tokenizer_en_pt_es_60000.json",
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    raise FileNotFoundError("Nenhum tokenizer compatível encontrado em artifacts/")


tokenizer_path = resolve_tokenizer_path()
tokenizer = tokenizers.Tokenizer.from_file(tokenizer_path)
print(f"Tokenizer carregado: {tokenizer_path}")

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

models = {}

for file in sorted(os.listdir(PATH_MODELS)):
    if not file.endswith(".pt") or not file.startswith("model_"):
        continue

    model_name = file.split(".")[0].replace("model_", "")
    try:
        version = int(model_name.lstrip("v")) if model_name.startswith("v") else None
    except ValueError:
        version = None

    if version is None or version < 80:
        continue

    config_path = os.path.join(PATH_CONFIGS, f"{model_name}_config.json")
    if not os.path.exists(config_path):
        print(f"Ignorando {model_name}: config não encontrado em {config_path}")
        continue

    with open(config_path, "r", encoding="utf-8") as f:
        config = json.load(f)

    model = Transformer(
        embedding_dim=config["embedding_dim"],
        encoder_num_layers=config["encoder_num_layers"],
        decoder_num_layers=config["decoder_num_layers"],
        encoder_hidden_dim=config["encoder_hidden_dim"],
        decoder_hidden_dim=config["decoder_hidden_dim"],
        encoder_num_heads=config["encoder_num_heads"],
        decoder_num_heads=config["decoder_num_heads"],
        encoder_dropout=config["encoder_dropout"],
        decoder_dropout=config["decoder_dropout"],
        vocab_size=config.get("vocab_size", 120000),
        kv_cache=True,
        pad_idx=0,
        separate_embedding=config.get("separate_embedding", False),
    )

    state = torch.load(os.path.join(PATH_MODELS, file), map_location=DEVICE)
    model.load_state_dict(state)
    model.to(DEVICE)
    model.eval()
    models[model_name] = model

print(f"Modelos Transformers carregados: {list(models.keys())}")

In [ ]:
frases = [
    "You've heard of the genie of the lamp?",
    "This is a test sentence for translation.",
    "The weather is beautiful today and the city is full of life.",
    "I am learning how to build neural machine translation models.",
    "She wants to travel to Brazil next summer.",
]

for model_name, model in models.items():
    print(f"\n=== Modelo: {model_name} ===")
    for frase in frases:
        src_ids = tokenizer.encode(frase).ids
        src_tensor = torch.tensor(src_ids, dtype=torch.long, device=DEVICE).unsqueeze(0)

        with torch.no_grad():
            translated = model.predict(
                src_tensor,
                bos_token_id=5,
                eos_token_id=6,
                max_len=300,
                reset_cache=True,
            )

        if isinstance(translated, list):
            tokens = translated[0] if translated and isinstance(translated[0], list) else translated
        elif isinstance(translated, torch.Tensor):
            tokens = translated[0].tolist()
        else:
            tokens = translated

        translated_text = tokenizer.decode(tokens, skip_special_tokens=False)
        print(f"Entrada: {frase}")
        print(f"Saída:   {translated_text}\n")